In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

from bellm.tokeniser import Tokeniser
from reader import get_dataset

/Users/belle/Developer/Belllm/belllm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = torch.device("mps")
EPOCHS = 50
BATCH_SIZE = 16
MAX_VOCAB = 5000
INPUT_CONTEXT_SIZE = 600
OUTPUT_CONTEXT_SIZE = 200

def tensor(x):
    return torch.tensor(x, dtype=torch.long, device=DEVICE)


In [3]:

class TextDiffusionTransformer(nn.Module):
    def __init__(self, vocab_size=10000, context_len=1000, diffuse_len=200, model_dim=512):
        super().__init__()

        self.diffuse_len = diffuse_len
        self.model_dim = model_dim

        # 1. Context Embedding (for the 1k fixed tokens)
        self.context_embedding = nn.Embedding(vocab_size, model_dim)

        # 2. Logit Projection (for the 200 tokens being diffused)
        # This takes the B x 200 x 10000 logits and brings them to model_dim
        self.diffuse_projection = nn.Linear(vocab_size, model_dim)

        # 3. Time Embedding (MLP)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, model_dim),
            nn.SiLU(),
            nn.Linear(model_dim, model_dim)
        )

        # 4. Transformer Layers
        # We use a standard Encoder; you can increase num_layers for better results
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=8,
            dim_feedforward=model_dim * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)

        # 5. Output Head (Back to 10k logits)
        self.to_velocity = nn.Linear(model_dim, vocab_size)

    def forward(self, context_ids, noisy_logits, t):
        """

        noisy_logits: (B, 200, 10000) - Continuous logit space
        t: (B, 1) - Timestep [0, 1]
        """
        # todo need positional embedding

        # Embed context
        c_emb = self.context_embedding(context_ids)  # (B, 1000, dim)

        # Project noisy logits
        d_emb = self.diffuse_projection(noisy_logits)  # (B, 200, dim)

        # Combine sequences (Total length: 1200)
        x = torch.cat([c_emb, d_emb], dim=1)

        # # Inject Time
        t_emb = self.time_mlp(t).unsqueeze(1)  # (B, 1, dim)
        x = x + t_emb

        # Process through Transformer
        x = self.transformer(x)

        # Slice out only the 200 "diffuse" tokens for the output
        diffuse_out = x[:, -self.diffuse_len:, :]

        pred_x0 = self.to_velocity(diffuse_out)
        # pred_x0 = F.softmax(pred_x0, dim=-1)

        # The velocity on the simplex is the straight line:
        # v = (pred_x0 - noise) / (1 - t)
        return pred_x0


In [11]:
model = TextDiffusionTransformer(
    vocab_size=MAX_VOCAB,
    context_len=INPUT_CONTEXT_SIZE,
    diffuse_len=OUTPUT_CONTEXT_SIZE
)

model.to(DEVICE)
print("Model Size:", sum(p.numel() for p in model.parameters()))

dataset = get_dataset()
tokeniser = Tokeniser().load(f"../src/bellm/tokeniser/tokeniser.json")

items = list(dataset \
     # .skip(i * SAMPLE_SIZE_BATCH_SIZE + start_offset) \
     .take(20_000)["text"])


Model Size: 26863496


In [12]:
import random

X, Y = [], []

for item in tqdm(items):
    tokens = tokeniser.tokenize(item, max_tokens=MAX_VOCAB).token_ids

    # for t_idx in range(1, len(tokens) - 1):

    t_idx = random.randint(0, len(tokens) - 1)

    input = tokens[:t_idx]
    output = tokens[t_idx + 1:]

    if len(input) < INPUT_CONTEXT_SIZE:
        input = [Tokeniser.PAD] * (INPUT_CONTEXT_SIZE - len(input)) + input

    if len(output) < OUTPUT_CONTEXT_SIZE:
        output = output + [Tokeniser.PAD] * (OUTPUT_CONTEXT_SIZE - len(output))

    if len(input) > INPUT_CONTEXT_SIZE:
        input = input[-INPUT_CONTEXT_SIZE:]

    # If the output is longer than context size, then truncate it and at a next page token
    if len(output) > OUTPUT_CONTEXT_SIZE:
        output = output[:OUTPUT_CONTEXT_SIZE - 1] + [Tokeniser.NEXT_PAGE]

    X.append(input)
    Y.append(output)

X = np.array(X)
Y = np.array(Y)

# Shuffle all input
shuffle_indexes = np.arange(len(X))
np.random.shuffle(shuffle_indexes)
X = torch.tensor(X[shuffle_indexes], dtype=torch.long)
Y = torch.tensor(Y[shuffle_indexes], dtype=torch.long)


100%|██████████| 20000/20000 [01:20<00:00, 247.70it/s]


In [13]:
Y.shape

torch.Size([20000, 200])

In [14]:
optimiser = torch.optim.AdamW(model.parameters())
loss = nn.CrossEntropyLoss()


for epoch in range(EPOCHS):
    model.train()
    losses = []

    for b_idx in range(0, len(X), BATCH_SIZE):
        batch_x = X[b_idx:b_idx + BATCH_SIZE].to(DEVICE)
        batch_y = Y[b_idx:b_idx + BATCH_SIZE].to(DEVICE)

        optimiser.zero_grad()
        noisy_input = torch.randn((len(batch_x), OUTPUT_CONTEXT_SIZE, MAX_VOCAB), dtype=torch.float)

        # Create the outputs the model should be generating in perfect world
        x0 = F.one_hot(batch_y, num_classes=MAX_VOCAB).float() # (B, 200, 10000)

        # 2. Create the "Noise" state (x1)
        # For a distribution, noise is usually a Uniform Distribution (Maximum Entropy)
        x1 = torch.ones_like(x0) / MAX_VOCAB

        # 3. Sample a random time t for each batch [0, 1]
        t = torch.rand(len(batch_x), 1, device=DEVICE)
        t_expanded = t.unsqueeze(-1) # (B, 1, 1) to match (B, 200, 10000)

        # 4. Construct the noisy flow xt
        # Linear interpolation between the clean one-hot and the uniform noise
        xt = (1 - t_expanded) * x0 + t_expanded * x1

        # 5. Model predicts the "Clean Logits"
        # Even though we feed in xt (probabilities), the model outputs
        # the 10k logits for the target x0.
        pred_logits = model(batch_x, xt, t) # (B, 200, 10000)

        # 6. Categorical Cross-Entropy Loss
        # This forces the model to recover the discrete IDs from the noisy mixture
        loss = F.cross_entropy(
            pred_logits.view(-1, MAX_VOCAB),
            batch_y.view(-1)
        )

        loss.backward()
        optimiser.step()

        l = loss.item()
        losses.append(l)

        print(f"\rEpoch: {epoch}. Batch: {b_idx}/{len(X)}.", str(np.mean(losses)), end="")
        batch_x = X[b_idx:b_idx + BATCH_SIZE].to("cpu")
        batch_y = Y[b_idx:b_idx + BATCH_SIZE].to("cpu")

    print(f"\rEpoch: {epoch}", str(np.mean(losses)))


Epoch: 0 5.54314109573364300. 5.5431410957336435
Epoch: 1 5.50600447235107500. 5.5060044723510755
Epoch: 2 5.50237994384765600. 5.5023799438476565
Epoch: 3 5.50028744430542000. 5.5002874443054274
Epoch: 4 5.49719363822937000. 5.4971936382293735
Epoch: 5 5.49544184265136700. 5.4954418426513675
Epoch: 6 5.49370909252166800. 5.4937090925216685
Epoch: 7 5.49091650314331050. 5.4909165031433105
Epoch: 8 5.48955928554534900. 5.4895592855453495
Epoch: 9 5.48806242790222100. 5.4880624279022215
Epoch: 10. Batch: 13520/20000. 5.5150293247074115

KeyboardInterrupt: 

In [16]:
model.eval()

text = "hello"
tokens = tokeniser.tokenize_batch([text])
model_input = torch.tensor(np.array([x.token_ids for x in tokens]), dtype=torch.long, device=DEVICE)


def generate_flow(model, context_ids, steps=20):
    B = context_ids.shape[0]
    # Start at t=1 (Pure Noise)
    xt = torch.randn(B, OUTPUT_CONTEXT_SIZE, MAX_VOCAB).to(context_ids.device)

    dt = 1.0 / steps

    for i in range(steps):
        # Current time (1.0 down to 0.0)
        t_val = 1.0 - (i * dt)
        t = torch.full((B, 1), t_val).to(context_ids.device)

        # Predict velocity
        v = model(context_ids, xt, t)

        # Euler Step: Update xt by moving in the direction of -velocity
        # (Minus because we are going from noise -> data)
        xt = xt - v * dt

    return xt # These are your denoised pre-softmax logits

softmax = F.softmax(generate_flow(model, model_input), dim=2)
tokens = torch.argmax(softmax, dim=2).detach().cpu().numpy()

print("".join(tokeniser.detokenise(tokens[0])))

nicationnicationnication significnication signific
 signific significnication


 signific
nication
 signific



 signific signific
nication signific signific
nication signific


 signific


 signific signific signific signific
nication




nication



 signific
 signific significnicationnicationnication

nication
 significnication signific
nication
 signific signific signific signific



nicationnicationnication signific



nication
 signific signific significnication




nicationnicationnication
nication



 signific
nicationnication
 significnication
 signific




 significnication



 significnicationnication

nicationnication
nicationnication
nication signific


 signific
nicationnication signific
nicationnication signific


nication




 signific

nication


nication significnication


nication
 signific significnication


nication



 significnicationnication
 signific


nication significnicationnicationnication


In [ ]:
tokens